## Settings

In [ ]:
## auto reload modules
%reload_ext autoreload
%autoreload 2

## Dependencies

In [ ]:
## libraries
import sys
import pickle
import logging
from pathlib import Path
from IPython.display import Markdown, display

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.data.builders import (
    load_processed_data,
    load_perturbed_data
)
from src.estimators.factories import load_estimators
from src.evaluators.caching import load_notebook_cache
from src.evaluators.helpers import spec_marginal_delta
from src.evaluators.tables import main_table
from src.evaluators.perturbing import (
    train_perturbed_transfer,
    train_perturbed_recovery,
    train_perturbed_consensus,
    compile_perturbed_transfer,
    compile_perturbed_recovery,
    compile_perturbed_consensus,
    compile_perturbed_full,
    find_perturbed_max,
    stat_perturbed_tost
)

## constants
from src.evaluators.config import CONSENSUS_METRICS
from src.evaluators.config import (
    FEAT_X,
    FEAT_Z,
    TARGET
)

## Initialization

In [ ]:
## reproducibility
N_DECIMALS = 2
N_REPEATS = 30
RANDOM_STATE = 42
FORCE_RECOMPUTE = False

## load data and models
_disable = logging.root.manager.disable
logging.disable(logging.INFO)
try:
    data_proc = load_processed_data()
    data_pert_all = load_perturbed_data()
    data_pert = {
        key: data_pert_all[key]
        for key in [
            "network_perturbed", 
            "invariants_perturbed", 
            "process_perturbed", 
            "signatures_perturbed"
        ]
        if key in data_pert_all
    }
finally:
    logging.disable(_disable)
    models = load_estimators(random_state = RANDOM_STATE)

## view data shape
n_obs, n_feat = data_proc.shape
print(f"Original Data: {n_feat} features, {n_obs} observations")
print("Perturbed Data:")
for json_key, methods in data_pert.items():
    print(f"  {json_key}:")
    for method, intense in methods.items():
        pert_df = next(iter(intense.values()))
        n_r, n_c = pert_df.shape
        print(f"   - {method}: {n_c} features, {n_r} observations")

## view model surface
n_mods = len(models)
print(f"Learned Models: {n_mods} estimators")

## cache setup
cache_path = root / "notebooks" / "cache" / "perturb_results.pkl"
cache_metadata = {
    "n_obs": len(data_proc),
    "n_repeats": N_REPEATS,
    "random_state": RANDOM_STATE,
    "model_names": sorted(models.keys()),
    "target": TARGET,
    "feat_x": list(FEAT_X),
    "feat_z": list(FEAT_Z),
}
cache_keys = (
    "results_perturbed_transfer",
    "results_perturbed_recovery",
    "results_perturbed_consensus",
)
cache_payload = load_notebook_cache(
    cache_path = cache_path,
    metadata = cache_metadata,
    required_keys = cache_keys,
    force_recompute = FORCE_RECOMPUTE,
)
if cache_payload is not None:
    globals().update({key: cache_payload[key] for key in cache_keys})

Original Data: 32 features, 25 observations
Perturbed Data:
  network_perturbed:
   - densify: 25 features, 634 observations
   - rewire: 25 features, 634 observations
   - sample: 25 features, 750 observations
  invariants_perturbed:
   - jitter: 25 features, 750 observations
   - noise: 25 features, 750 observations
   - subset: 25 features, 750 observations
  process_perturbed:
   - bootstrapping: 11 features, 750 observations
   - scaling: 11 features, 25 observations
   - smoothing: 11 features, 25 observations
  signatures_perturbed:
   - jitter: 11 features, 750 observations
   - noise: 11 features, 750 observations
   - subset: 11 features, 750 observations
Learned Models: 9 estimators
Loaded results from /Users/nickkunz/Documents/Nick’s MacBook Pro/Python/network-capacity/notebooks/cache/perturb_results.pkl


## Training

In [ ]:
## perturbation transfer training
if cache_payload is None:
    results_dict_perturbed_transfer = train_perturbed_transfer(
        data = data_proc,
        models = models,
        data_pert = data_pert,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
    )

In [ ]:
## perturbation recovery training
if cache_payload is None:
    results_dict_perturbed_recovery = train_perturbed_recovery(
        data = data_proc,
        models = models,
        data_pert = data_pert,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        n_repeats = N_REPEATS,
        random_state = RANDOM_STATE,
    )

In [ ]:
## perturbation pairwise consensus training
if cache_payload is None:
    results_dict_perturbed_consensus = train_perturbed_consensus(
        data = data_proc,
        models = models,
        data_pert = data_pert,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        n_repeats = N_REPEATS,
        random_state = RANDOM_STATE,
    )

## Post-Processing

In [ ]:
if cache_payload is None:
    results_perturbed_transfer, recovery_perturbed_transfer = compile_perturbed_transfer(
        results = results_dict_perturbed_transfer,
    )
    results_perturbed_recovery = compile_perturbed_recovery(
        results = results_dict_perturbed_recovery,
    )

if cache_payload is None or "results_perturbed_full_agreement" not in cache_payload:
    if cache_payload is not None:
        results_dict_perturbed_consensus = train_perturbed_consensus(
            data = data_proc,
            models = models,
            data_pert = data_pert,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            target = TARGET,
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE,
            n_jobs = -1,
        )
    results_perturbed_consensus = compile_perturbed_consensus(
        results = results_dict_perturbed_consensus,
    )
    results_perturbed_full_agreement = compile_perturbed_full(
        results = results_dict_perturbed_consensus,
        data = data_proc,
        target = TARGET,
    )
    cache_path.parent.mkdir(parents = True, exist_ok = True)
    cache_payload = {
        **(cache_payload or {}),
        "metadata": cache_metadata,
        "results_perturbed_transfer": results_perturbed_transfer,
        "results_perturbed_recovery": results_perturbed_recovery,
        "results_perturbed_consensus": results_perturbed_consensus,
        "results_perturbed_full_agreement": results_perturbed_full_agreement,
    }
    cache_path.write_bytes(pickle.dumps(cache_payload))
    print(f"Cached perturbation results -> {cache_path}")

## Domain Transfer Equivalence Test
The domain transfer test probes whether perturbations preserve EI transfer performance. Predictions are generated under the same frozen LOGO-CV (domain) protocol and scored by EI per (model x domain). Each perturbation method is tested for statistical equivalence against the original baseline EI using two one-sided tests (TOST) with paired Wilcoxon signed-rank statistics.

- **$H_0$**: The absolute difference meets or exceeds the margin ($|\Delta EI| \ge \delta$).
- **$H_1$**: The absolute difference falls strictly within the margin ($|\Delta EI| < \delta$).

Rejecting $H_0$ establishes that the perturbation effect is negligibly small relative to $\delta$.

### Maximum-Intensity Protocol
Only the highest intensity setting per perturbation method enters the test. Baseline and perturbed EI values are paired on (model $\times$ domain), matching the prediction consensus test. This yields one paired comparison per model $\times$ domain $\times$ method and prevents a dense intensity sweep from inflating sample size.

### Empirical Test Specification
The equivalence margin $\delta$ is derived from baseline variability across learners on the original EI transfer results. The full interquartile range of the baseline EI values, floored to the second decimal, sets $\delta$, anchoring the margin to natural performance dispersion rather than to perturbation effects.

In [ ]:
## maximum intensity per perturbation method
results_perturbed_transfer_max = find_perturbed_max(
    results = results_perturbed_transfer
)

## empirical delta specification
delta_transfer = spec_marginal_delta(
    results = results_perturbed_transfer,
    feat_value = ["ei"],
    track = "frozen",
    label_ref = "perturbation",
    method = "iqr",
    scale = 1.00,  ## full iqr range
    decimals = N_DECIMALS
)

In [ ]:
## tost equivalence test across all perturbation methods
results_transfer = stat_perturbed_tost(
    results = results_perturbed_transfer_max,
    feat_value = ["ei"],
    feat_group = ["perturbation", "method"],
    track = "frozen",
    delta = delta_transfer,
    decimals = N_DECIMALS
)

display(results_transfer)

Paired TOST (Wilcoxon Signed-Rank): n = 45, δ = 0.08
H₀: |Δ EI| ≥ δ
H₁: |Δ EI| < δ
Median Δ EI: Median of paired differences (perturbed - original), not the difference of marginal medians
Rank-biserial r: Raw paired effect size; positive values indicate perturbed > original, independent of TOST
TOST p: max(Upper p, Lower p)
Holm-adj. p: Holm-Bonferroni adjusted TOST p-value
Significance codes reflect Holm-adj. p
*** p < 0.001, ** p < 0.01, * p < 0.05


Median Δ EI Rank-biserial r  TOST p Holm-adj. p  \
Perturbation Method                                                          
invariants   jitter               0.00           -0.45  <0.001      <0.001   
             noise                0.00           -0.60  <0.001      <0.001   
             subset               0.00            0.05  <0.001      <0.001   
network      densify              0.00           -0.40  <0.001      <0.001   
             rewire               0.00           -0.23  <0.001      <0.001   
             sample               0.00           -0.35  <0.001      <0.001   
process      bootstrapping        0.01            0.33  <0.001      <0.001   
             scaling             -0.01           -0.45  <0.001      <0.001   
             smoothing           -0.02           -0.75  <0.001      <0.001   
signatures   jitter              -0.00           -0.31  <0.001      <0.001   
             noise               -0.00           -0.44  <0.001      <0.001   
             subset              -0.01           -0.61  <0.001      <0.001   

                           Sig.  Eq.  
Perturbation Method                   
invariants   jitter         ***  Yes  
             noise          ***  Yes  
             subset         ***  Yes  
network      densify        ***  Yes  
             rewire         ***  Yes  
             sample         ***  Yes  
process      bootstrapping  ***  Yes  
             scaling        ***  Yes  
             smoothing      ***  Yes  
signatures   jitter         ***  Yes  
             noise          ***  Yes  
             subset         ***  Yes

_All twelve perturbation methods across the four categories satisfy TOST equivalence at maximum intensity within $\delta_{EI} = 0.08$. Median EI shifts are uniformly small, with the largest change (−0.02, process smoothing) at one-quarter of the empirically derived margin and all Holm-adjusted TOST $p$-values below 0.001. These results indicate that domain-transfer performance is stable across substantial perturbations to network structure, graph invariant representations, stochastic process structure, and process signature representations, supporting robustness within the tested equivalence design._


## Prediction Consensus Equivalence Test
The prediction consensus test probes whether predictions preserve the observed capacity ordering. Predictions are generated under the same frozen LOGO-CV (domain) protocol, then scored by the Consensus Index CI = f(ρ, RBO, dCor) per (model x domain). Each perturbation method is tested for statistical equivalence against the original baseline CI using two one-sided tests (TOST) with paired Wilcoxon signed-rank statistics.

- **$H_0$**: The absolute difference meets or exceeds the margin ($|\Delta CI| \ge \delta$).
- **$H_1$**: The absolute difference falls strictly within the margin ($|\Delta CI| < \delta$).

Rejecting $H_0$ establishes that the perturbation effect is negligibly small relative to $\delta$.

### Maximum-Intensity Protocol
Only the highest intensity setting per perturbation method enters the test. Baseline and perturbed CI values are paired on (model $\times$ domain), matching the transfer EI test. This yields one paired comparison per model $\times$ domain $\times$ method and prevents a dense intensity sweep from inflating sample size.

### Empirical Test Specification
The equivalence margin $\delta$ is derived from baseline variability across learners in the original prediction consensus results. The full interquartile range of the baseline CI values, floored to the second decimal, sets $\delta$, anchoring the margin to natural performance dispersion rather than to perturbation effects.


In [ ]:
## prediction consensus equivalence at maximum perturbation intensity
results_perturbed_recovery_max = find_perturbed_max(
    results = results_perturbed_recovery
)

## empirical delta specification
delta_recovery = spec_marginal_delta(
    results = results_perturbed_recovery,
    feat_value = ["ci"],
    label_pert = "perturbation",
    track = "frozen",
    method = "iqr",
    scale = 1.00,  ## full iqr range
    decimals = N_DECIMALS
)

## tost equivalence test across all perturbation methods
results_recovery = stat_perturbed_tost(
    results = results_perturbed_recovery_max,
    feat_value = ["ci"],
    feat_group = ["perturbation", "method"],
    track = "frozen",
    delta = delta_recovery,
    decimals = N_DECIMALS
)

display(results_recovery)

Paired TOST (Wilcoxon Signed-Rank): n = 45, δ = 0.14
H₀: |Δ CI| ≥ δ
H₁: |Δ CI| < δ
Median Δ CI: Median of paired differences (perturbed - original), not the difference of marginal medians
Rank-biserial r: Raw paired effect size; positive values indicate perturbed > original, independent of TOST
TOST p: max(Upper p, Lower p)
Holm-adj. p: Holm-Bonferroni adjusted TOST p-value
Significance codes reflect Holm-adj. p
*** p < 0.001, ** p < 0.01, * p < 0.05


Median Δ CI Rank-biserial r  TOST p Holm-adj. p  \
Perturbation Method                                                          
invariants   jitter               0.00           -0.44  <0.001      <0.001   
             noise                0.00           -0.42  <0.001      <0.001   
             subset               0.00           -0.41  <0.001      <0.001   
network      densify              0.00           -0.47  <0.001      <0.001   
             rewire               0.00           -0.43  <0.001      <0.001   
             sample               0.00           -0.58  <0.001      <0.001   
process      bootstrapping       -0.11           -0.82   0.028       0.028   
             scaling             -0.02           -0.23  <0.001      <0.001   
             smoothing           -0.06           -0.73   0.002       0.003   
signatures   jitter              -0.01           -0.29  <0.001      <0.001   
             noise               -0.01           -0.68  <0.001      <0.001   
             subset              -0.04           -0.82  <0.001      <0.001   

                           Sig.  Eq.  
Perturbation Method                   
invariants   jitter         ***  Yes  
             noise          ***  Yes  
             subset         ***  Yes  
network      densify        ***  Yes  
             rewire         ***  Yes  
             sample         ***  Yes  
process      bootstrapping    *  Yes  
             scaling        ***  Yes  
             smoothing       **  Yes  
signatures   jitter         ***  Yes  
             noise          ***  Yes  
             subset         ***  Yes

_All twelve perturbation methods across the four categories satisfy TOST equivalence for prediction consensus at maximum intensity within $\delta_{CI} = 0.14$. Median CI shifts range from 0.00 for the network and invariant perturbations to −0.11 for process bootstrapping, with Holm-adjusted TOST $p$-values at most 0.025, indicating that the observed capacity ordering is preserved despite substantial perturbations to network structure, invariant representations, stochastic process structure, and process signature representations. The prediction consensus signal is therefore robust within the tested equivalence design._


## Pairwise Prediction Consensus Evaluation
The pairwise prediction consensus evaluation probes whether maximum-intensity perturbations preserve agreement among independently trained frontiers. Unlike the domain-transfer and prediction-consensus sections, this is a descriptive stability analysis rather than a formal equivalence test. Agreement is computed across all model pairs using the same consensus metrics.

- **Spearman (`ρ`)**: Global rank agreement between paired model frontiers.
- **Rank-Biased Overlap (`RBO`)**: Top-weighted agreement, emphasizing the high-capacity tail.
- **Distance Correlation (`dCor`)**: Nonlinear dependence between paired model frontiers.
- **Consensus Index (`CI`)**: Geometric mean of agreement-scaled `ρ`, `RBO`, and `dCor`.

### Maximum-Intensity Protocol
Only the highest intensity setting per perturbation method enters the evaluation. The unperturbed baseline is retained for comparison, and perturbed values are summarized across model pairs within each method.

### Reporting Convention
Each perturbation method contributes one value per model pair at maximum intensity. The preview reports medians across all retained pairwise comparisons. Prior to compounding, `ρ` is rescaled from `[-1, 1]` to `[0, 1]` so that negative rank agreement lowers `CI` without collapsing to a single floor. Higher values indicate stronger perturbation-stable pairwise prediction consensus.

In [ ]:
## preview results
results_perturbed_consensus_max = find_perturbed_max(
    results = results_perturbed_consensus
)
results_perturbed_consensus_max.loc[
    results_perturbed_consensus_max["perturbation"] != "baseline"
][CONSENSUS_METRICS].median()

rho    0.821538
rbo    0.821364
dcr    0.838811
ci     0.857427
dtype: float64

_Maximum-intensity perturbations preserve substantial pairwise prediction consensus, with median `CI` = 0.86 across retained pairwise comparisons. Median `ρ`, `RBO`, and `dCor` remain uniformly high, indicating that different learning paradigms continue to agree on the capacity ordering after strong interventions. The shared ordering signal therefore persists across perturbations._


## Summary

In [ ]:
## manuscript master table for perturbation robustness
display_table = main_table(
    results_left = results_transfer.reset_index(),
    results_right = results_recovery.reset_index(),
    index_feat = ["Perturbation", "Method"],
    value_feat_left = ["Median Δ EI", "Rank-biserial r", "Sig.", "Eq."],
    value_feat_right = ["Median Δ CI", "Rank-biserial r", "Sig.", "Eq."],
    header_left = "Domain Transfer",
    header_right = "Estimation-Observation Consensus",
    short_feat_left = ["Δ EI", "r", "Sig.", "Eq."],
    short_feat_right = ["Δ CI", "r", "Sig.", "Eq."],
    row_order = [
        ("network", "sample"), ("network", "rewire"), ("network", "densify"),
        ("invariants", "jitter"), ("invariants", "noise"), ("invariants", "subset"),
        ("process", "bootstrapping"), ("process", "scaling"), ("process", "smoothing"),
        ("signatures", "jitter"), ("signatures", "noise"), ("signatures", "subset"),
    ],
)

display(display_table)
display(
    Markdown(
        "**Δ EI**, **Δ CI**: median paired difference between perturbed and original performance at maximum perturbation intensity, where values near zero indicate performance preservation. "
        "Positive values indicate perturbed > original and negative values indicate perturbed < original. "
        "**r**: rank-biserial correlation effect size. "
        "**Sig.**: Holm-adjusted TOST *p* below threshold to test within each outcome "
        "(\*\*\* *p* < 0.001, \*\* *p* < 0.01, \* *p* < 0.05); significance supports the equivalence hypothesis "
        "H₁: |Δ| < δ, that original data has equivalent performance to perturbed data. "
        "**Eq.**: equivalence supported when Holm-adjusted TOST *p* < 0.05. "
        "Equivalence margin δ = IQR derived from natural variability from the original data. "
        "Each test is conducted on *n* = 45 (9 models × 5 domains)."
    )
)

Domain Transfer                   \
                                      Δ EI      r Sig.  Eq.   
Perturbation Method                                           
network      sample                   0.00  -0.35  ***  Yes   
             rewire                   0.00  -0.23  ***  Yes   
             densify                  0.00  -0.40  ***  Yes   
invariants   jitter                   0.00  -0.45  ***  Yes   
             noise                    0.00  -0.60  ***  Yes   
             subset                   0.00   0.05  ***  Yes   
process      bootstrapping            0.01   0.33  ***  Yes   
             scaling                 -0.01  -0.45  ***  Yes   
             smoothing               -0.02  -0.75  ***  Yes   
signatures   jitter                  -0.00  -0.31  ***  Yes   
             noise                   -0.00  -0.44  ***  Yes   
             subset                  -0.01  -0.61  ***  Yes   

                           Estimation-Observation Consensus                   
                                                       Δ CI      r Sig.  Eq.  
Perturbation Method                                                           
network      sample                                    0.00  -0.58  ***  Yes  
             rewire                                    0.00  -0.43  ***  Yes  
             densify                                   0.00  -0.47  ***  Yes  
invariants   jitter                                    0.00  -0.44  ***  Yes  
             noise                                     0.00  -0.42  ***  Yes  
             subset                                    0.00  -0.41  ***  Yes  
process      bootstrapping                            -0.11  -0.82    *  Yes  
             scaling                                  -0.02  -0.23  ***  Yes  
             smoothing                                -0.06  -0.73   **  Yes  
signatures   jitter                                   -0.01  -0.29  ***  Yes  
             noise                                    -0.01  -0.68  ***  Yes  
             subset                                   -0.04  -0.82  ***  Yes

**Δ EI**, **Δ CI**: median paired difference between perturbed and original performance at maximum perturbation intensity, where values near zero indicate performance preservation. Positive values indicate perturbed > original and negative values indicate perturbed < original. **r**: rank-biserial correlation effect size. **Sig.**: Holm-adjusted TOST *p* below threshold to test within each outcome (\*\*\* *p* < 0.001, \*\* *p* < 0.01, \* *p* < 0.05); significance supports the equivalence hypothesis H₁: |Δ| < δ, that original data has equivalent performance to perturbed data. **Eq.**: equivalence supported when Holm-adjusted TOST *p* < 0.05. Equivalence margin δ = IQR derived from natural variability from the original data. Each test is conducted on *n* = 45 (9 models × 5 domains).